# Limpeza dos dados — Despesas SMS Goiânia (2025–2026)

Consolida os 18 arquivos brutos de `data/raw/2025/` e `data/raw/2026/` em uma única base
tratada, pronta para a análise exploratória (feita à parte, no Sheets).

Escopo apenas de limpeza — sem agregações de negócio (Pareto, série mensal, outliers) e sem
gráficos. O esquema das 22 colunas originais, com tipos e sujeiras conhecidas, está documentado
em `docs/dicionario-dados.md`. As decisões tomadas aqui são resumidas em `docs/log-limpeza.md`.

## Etapa 1 — Setup

Confere se o `pandas` está disponível (instala se não estiver) e que a estrutura de pastas
descrita no README existe: `data/raw/2025/`, `data/raw/2026/` (brutos, intocados) e `data/clean/`
(saída desta limpeza).

In [1]:
import subprocess
import sys

try:
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd

from pathlib import Path

print("pandas:", pd.__version__)

pandas: 3.0.5


In [2]:
RAW_2025 = Path("../data/raw/2025")
RAW_2026 = Path("../data/raw/2026")
CLEAN_DIR = Path("../data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

for pasta in (RAW_2025, RAW_2026, CLEAN_DIR):
    print(pasta, "->", "existe" if pasta.exists() else "NAO EXISTE")

print("import pandas OK, estrutura de pastas confere com o README.")

..\data\raw\2025 -> existe
..\data\raw\2026 -> existe
..\data\clean -> existe
import pandas OK, estrutura de pastas confere com o README.


## Etapa 2 — Leitura de todos os arquivos mensais

Os 18 arquivos são lidos com `sep=';'`, `encoding='latin-1'` (o TransWeb não exporta em UTF-8) e
`decimal=','`.

Todas as colunas são lidas como texto (`dtype=str`) nesta etapa. Isso é proposital: várias
colunas que parecem numéricas são, na verdade, identificadores (`Empenho`, `Liquidacao`,
`OrdemPagamento`, `CNPJ`, `NumeroLicitacao`) — se o pandas inferir o tipo automaticamente, zeros à
esquerda somem e IDs longos (`OrdemPagamento` tem 22 dígitos) perdem precisão ao virar `float64`.
A conversão de tipos "de verdade" (datas e valores) fica só na Etapa 6, de forma explícita e
auditável.

Cada arquivo vira um DataFrame com uma coluna auxiliar `arquivo_origem` (rastreio de onde cada
linha veio — não entra em nenhuma agregação).

In [3]:
arquivos = sorted(RAW_2025.glob("*.csv")) + sorted(RAW_2026.glob("*.csv"))
assert len(arquivos) == 18, f"esperado 18 arquivos, encontrado {len(arquivos)}"

dfs_brutos = []
for arq in arquivos:
    df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",", dtype=str)
    df["arquivo_origem"] = arq.name
    dfs_brutos.append(df)

for arq, df in zip(arquivos, dfs_brutos):
    print(f"{arq.parent.name}/{arq.name}: {len(df)} linhas (bruto, com rodape)")

2025/Despesas_20250101_20250131.csv: 1300 linhas (bruto, com rodape)
2025/Despesas_20250201_20250228.csv: 604 linhas (bruto, com rodape)
2025/Despesas_20250301_20250331.csv: 647 linhas (bruto, com rodape)
2025/Despesas_20250401_20250430.csv: 1453 linhas (bruto, com rodape)
2025/Despesas_20250501_20250531.csv: 539 linhas (bruto, com rodape)
2025/Despesas_20250601_20250630.csv: 559 linhas (bruto, com rodape)
2025/Despesas_20250701_20250731.csv: 778 linhas (bruto, com rodape)
2025/Despesas_20250801_20250831.csv: 625 linhas (bruto, com rodape)
2025/Despesas_20250901_20250930.csv: 802 linhas (bruto, com rodape)
2025/Despesas_20251001_20251031.csv: 646 linhas (bruto, com rodape)
2025/Despesas_20251101_20251130.csv: 320 linhas (bruto, com rodape)
2025/Despesas_20251201_20251231.csv: 329 linhas (bruto, com rodape)
2026/Despesas_20260101_20260131.csv: 1857 linhas (bruto, com rodape)
2026/Despesas_20260201_20260228.csv: 1162 linhas (bruto, com rodape)
2026/Despesas_20260301_20260331.csv: 705 lin

In [4]:
print(f"Está bom quando: 18 arquivos carregados sem erro de encoding.")
print(f"Arquivos lidos: {len(dfs_brutos)} de 18 esperados.")
print(f"Total de linhas brutas (com rodape): {sum(len(d) for d in dfs_brutos)}")
assert len(dfs_brutos) == 18
print("OK — todos os arquivos carregaram sem KeyError/UnicodeDecodeError.")

Está bom quando: 18 arquivos carregados sem erro de encoding.
Arquivos lidos: 18 de 18 esperados.
Total de linhas brutas (com rodape): 13308
OK — todos os arquivos carregaram sem KeyError/UnicodeDecodeError.


## Etapa 3 — Remoção do rodapé

Cada arquivo termina com uma linha separadora (`------`) e três linhas de totalização
(`Total Empenhado`, `Total Liquidado`, `Total Pago`) — ver `docs/dicionario-dados.md`, seção 3.

A regra já validada no Sheets (`df.iloc[:, 0].astype(str).str.startswith('-')`) só remove a linha
separadora: as três linhas de "Total ..." **não** começam com `-`, então passam ilesas por esse
filtro sozinho. Por isso a checagem extra do prompt não é opcional na prática — ela é o que de
fato descarta as linhas de totalização: depois do filtro de `-`, exige-se que `Empenho` seja uma
sequência de dígitos.

Nota: o dicionário de dados registra `Empenho` como tendo 18 dígitos, mas o formato observado nos
arquivos reais é de **17 dígitos** (ex.: `20252150000630001`). Usamos o formato real (17) — o
dicionário deveria ser corrigido.

In [5]:
import re

EMPENHO_RE = re.compile(r"^\d{17}$")

dfs_sem_rodape = []
for arq, df in zip(arquivos, dfs_brutos):
    sem_separador = df[~df.iloc[:, 0].astype(str).str.startswith("-")]
    valido = sem_separador["Empenho"].astype(str).str.fullmatch(EMPENHO_RE)
    limpo = sem_separador[valido]
    dfs_sem_rodape.append(limpo)
    n_removido = len(df) - len(limpo)
    print(f"{arq.name}: {len(df)} -> {len(limpo)} linhas ({n_removido} de rodape removidas)")

Despesas_20250101_20250131.csv: 1300 -> 1296 linhas (4 de rodape removidas)
Despesas_20250201_20250228.csv: 604 -> 600 linhas (4 de rodape removidas)
Despesas_20250301_20250331.csv: 647 -> 643 linhas (4 de rodape removidas)
Despesas_20250401_20250430.csv: 1453 -> 1449 linhas (4 de rodape removidas)
Despesas_20250501_20250531.csv: 539 -> 535 linhas (4 de rodape removidas)
Despesas_20250601_20250630.csv: 559 -> 555 linhas (4 de rodape removidas)
Despesas_20250701_20250731.csv: 778 -> 774 linhas (4 de rodape removidas)
Despesas_20250801_20250831.csv: 625 -> 621 linhas (4 de rodape removidas)
Despesas_20250901_20250930.csv: 802 -> 798 linhas (4 de rodape removidas)
Despesas_20251001_20251031.csv: 646 -> 642 linhas (4 de rodape removidas)
Despesas_20251101_20251130.csv: 320 -> 316 linhas (4 de rodape removidas)
Despesas_20251201_20251231.csv: 329 -> 325 linhas (4 de rodape removidas)
Despesas_20260101_20260131.csv: 1857 -> 1853 linhas (4 de rodape removidas)
Despesas_20260201_20260228.csv: 

In [6]:
# Checagem manual antes/depois em 3 arquivos: mostra as ultimas linhas originais
# (deveriam incluir separador + 3 totais) contra as ultimas linhas depois da limpeza
# (deveriam ser so despesas de verdade, com Empenho de 17 digitos).
for i in [0, 5, 17]:
    arq, bruto, limpo = arquivos[i], dfs_brutos[i], dfs_sem_rodape[i]
    print("=" * 70)
    print(arq.name)
    print("--- ultimas 4 linhas ANTES (deve conter separador + 3 totais) ---")
    print(bruto.iloc[-4:, 0].tolist())
    print("--- ultima linha DEPOIS (deve ser um Empenho de 17 digitos) ---")
    print(limpo.iloc[-1:, 0].tolist())

Despesas_20250101_20250131.csv
--- ultimas 4 linhas ANTES (deve conter separador + 3 totais) ---
['-------------------------------------------', 'Total Empenhado', 'Total Liquidado ', 'Total Pago']
--- ultima linha DEPOIS (deve ser um Empenho de 17 digitos) ---
['20252150047110014']
Despesas_20250601_20250630.csv
--- ultimas 4 linhas ANTES (deve conter separador + 3 totais) ---
['-------------------------------------------', 'Total Empenhado', 'Total Liquidado ', 'Total Pago']
--- ultima linha DEPOIS (deve ser um Empenho de 17 digitos) ---
['20252150048110001']
Despesas_20260601_20260630.csv
--- ultimas 4 linhas ANTES (deve conter separador + 3 totais) ---
['-------------------------------------------', 'Total Empenhado', 'Total Liquidado ', 'Total Pago']
--- ultima linha DEPOIS (deve ser um Empenho de 17 digitos) ---
['20262150045330002']


In [7]:
total_removido = sum(len(b) - len(l) for b, l in zip(dfs_brutos, dfs_sem_rodape))
print(f"Está bom quando: nenhuma linha de totalização sobra.")
print(f"Total de linhas de rodape removidas nos 18 arquivos: {total_removido}")
print(f"Esperado: 4 linhas por arquivo (1 separador + 3 totais) x 18 = 72")

for arq, limpo in zip(arquivos, dfs_sem_rodape):
    resto = limpo["Empenho"].astype(str).str.fullmatch(EMPENHO_RE)
    assert resto.all(), f"{arq.name} ainda tem linha de rodape"

print("OK — toda linha restante em todos os 18 arquivos tem Empenho valido (17 digitos).")

Está bom quando: nenhuma linha de totalização sobra.
Total de linhas de rodape removidas nos 18 arquivos: 72
Esperado: 4 linhas por arquivo (1 separador + 3 totais) x 18 = 72
OK — toda linha restante em todos os 18 arquivos tem Empenho valido (17 digitos).


## Etapa 4 — Seleção de colunas

Remoção de `Objeto`, `NmOrgao` e `UnidadeOrcamentaria`:

- `Objeto` — texto livre que mistura histórico do empenho com dados pessoais (CPF, dados
  bancários de beneficiários), publicados pela fonte oficial mas que este projeto optou por não
  redistribuir (compromisso de LGPD do README).
- `NmOrgao` e `UnidadeOrcamentaria` — constantes em todo o dataset (sempre
  `SECRETARIA MUNICIPAL DE SAUDE`, já que a coleta filtrou por esse órgão); não agregam
  informação e `UnidadeOrcamentaria` ainda vem com padding de espaços.

O dicionário de dados registra o nome de coluna como `NmOrgao` (não `NomeOrgao`) — conferido
contra o cabeçalho real dos 18 arquivos antes de rodar o `drop`, já que o export do TransWeb pode
variar.

In [8]:
colunas_esperadas = set(dfs_sem_rodape[0].columns) - {"arquivo_origem"}
for arq, df in zip(arquivos, dfs_sem_rodape):
    cols = set(df.columns) - {"arquivo_origem"}
    assert cols == colunas_esperadas, f"{arq.name} tem cabecalho diferente: {cols ^ colunas_esperadas}"
print("Cabecalho identico nos 18 arquivos. Colunas encontradas:")
print(sorted(colunas_esperadas))

COLUNAS_REMOVIDAS = ["Objeto", "NmOrgao", "UnidadeOrcamentaria"]
assert set(COLUNAS_REMOVIDAS) <= colunas_esperadas, "nome de coluna nao bate com o cabecalho real"

Cabecalho identico nos 18 arquivos. Colunas encontradas:
['CNPJ', 'DataEmpenho', 'DataLiquidacao', 'DataPagamento', 'DsNaturezadaDespesa', 'Empenho', 'FonteRecurso', 'Funcao', 'Liquidacao', 'Modalidade', 'NaturezaDespesa', 'NmFavorecido', 'NmOrgao', 'NumeroLicitacao', 'Objeto', 'OrdemPagamento', 'SubFuncao', 'UnidadeOrcamentaria', 'VlAnulado', 'VlEmpenhado', 'VlLiquidado', 'VlPago']


In [9]:
dfs_colunas_ok = [df.drop(columns=COLUNAS_REMOVIDAS) for df in dfs_sem_rodape]

print(f"Está bom quando: a base final nao carrega 'Objeto' e sobram so colunas uteis.")
print(f"Colunas antes: {len(dfs_sem_rodape[0].columns)}")
print(f"Colunas depois: {len(dfs_colunas_ok[0].columns)}")
assert "Objeto" not in dfs_colunas_ok[0].columns
print("OK — 'Objeto' removido; colunas restantes:")
print(dfs_colunas_ok[0].columns.tolist())

Está bom quando: a base final nao carrega 'Objeto' e sobram so colunas uteis.
Colunas antes: 23
Colunas depois: 20
OK — 'Objeto' removido; colunas restantes:
['Empenho', 'DataEmpenho', 'VlEmpenhado', 'Liquidacao', 'DataLiquidacao', 'VlLiquidado', 'OrdemPagamento', 'DataPagamento', 'VlPago', 'NaturezaDespesa', 'DsNaturezadaDespesa', 'CNPJ', 'VlAnulado', 'Funcao', 'SubFuncao', 'FonteRecurso', 'NmFavorecido', 'Modalidade', 'NumeroLicitacao', 'arquivo_origem']


## Etapa 5 — Concatenação e deduplicação

Concatena os 18 DataFrames já sem rodapé e sem as colunas removidas. A deduplicação é feita
sobre todas as colunas **exceto** `arquivo_origem` (é só rastreio, não deveria por si só tornar
duas linhas "diferentes" — na prática, como cada duplicata exata ocorre dentro do mesmo arquivo,
o resultado é idêntico incluindo ou não essa coluna, mas a exclusão é a regra correta).

In [10]:
base = pd.concat(dfs_colunas_ok, ignore_index=True)
print(f"Linhas apos concatenar os 18 arquivos: {len(base)}")

colunas_dedup = [c for c in base.columns if c != "arquivo_origem"]
n_duplicatas = base.duplicated(subset=colunas_dedup).sum()
print(f"Duplicatas exatas encontradas (antes de remover): {n_duplicatas}")

base = base.drop_duplicates(subset=colunas_dedup).reset_index(drop=True)
print(f"Linhas apos remover duplicatas: {len(base)}")

Linhas apos concatenar os 18 arquivos: 13236
Duplicatas exatas encontradas (antes de remover): 256
Linhas apos remover duplicatas: 12980


In [11]:
print("Está bom quando: o numero de duplicatas bate (ou explica divergencia) com a referencia de ~242.")
print(f"Duplicatas totais (2025+2026): {n_duplicatas}")
print()
print("Nota: a referencia de ~242 duplicatas foi levantada no Sheets so com os 12 arquivos de 2025.")
print("Rodando a mesma checagem apenas nos arquivos de 2025 (antes de juntar 2026):")

dedup_2025_only = pd.concat([d for d, a in zip(dfs_colunas_ok, arquivos) if '2025' in a.name], ignore_index=True)
n_dup_2025 = dedup_2025_only.duplicated(subset=colunas_dedup).sum()
print(f"  duplicatas so em 2025: {n_dup_2025} (referencia: ~242)")
print(f"  duplicatas adicionais trazidas por 2026: {n_duplicatas - n_dup_2025}")
assert n_dup_2025 == 242, "divergencia da referencia de 2025 precisa ser investigada"
print("OK — 242 duplicatas de 2025 batem exatamente com a referencia; as 14 duplicatas")
print("adicionais vêm dos 6 arquivos de 2026, que não faziam parte da checagem original no Sheets.")

Está bom quando: o numero de duplicatas bate (ou explica divergencia) com a referencia de ~242.
Duplicatas totais (2025+2026): 256

Nota: a referencia de ~242 duplicatas foi levantada no Sheets so com os 12 arquivos de 2025.
Rodando a mesma checagem apenas nos arquivos de 2025 (antes de juntar 2026):


  duplicatas so em 2025: 242 (referencia: ~242)
  duplicatas adicionais trazidas por 2026: 14
OK — 242 duplicatas de 2025 batem exatamente com a referencia; as 14 duplicatas
adicionais vêm dos 6 arquivos de 2026, que não faziam parte da checagem original no Sheets.


## Etapa 6 — Tipos e valores

- Datas (`DataEmpenho`, `DataLiquidacao`, `DataPagamento`) → `datetime`, no formato
  `dd/mm/aaaa HH:MM:SS` da fonte.
- Valores (`VlEmpenhado`, `VlLiquidado`, `VlPago`, `VlAnulado`) → `float`, trocando a vírgula
  decimal por ponto (não há separador de milhar nos dados, conforme `docs/dicionario-dados.md`).
- `DataLiquidacao`/`DataPagamento` nulas são esperadas: representam empenhos ainda não liquidados
  ou pagos, não erro de dado — por isso a validação abaixo compara o número de nulos **antes** e
  **depois** da conversão, em vez de simplesmente proibir `NaT`.
- Linhas com `VlPago = 0` são mantidas (compromisso empenhado/liquidado, ainda não pago) e
  sinalizadas com a coluna booleana `pago`.

In [12]:
COLUNAS_DATA = ["DataEmpenho", "DataLiquidacao", "DataPagamento"]
COLUNAS_VALOR = ["VlEmpenhado", "VlLiquidado", "VlPago", "VlAnulado"]

nulos_antes = {c: base[c].isna().sum() for c in COLUNAS_DATA}

for c in COLUNAS_DATA:
    base[c] = pd.to_datetime(base[c], format="%d/%m/%Y %H:%M:%S", errors="coerce")

for c in COLUNAS_VALOR:
    base[c] = (
        base[c]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

base["pago"] = base["VlPago"] > 0

print(base.dtypes)

Empenho                           str
DataEmpenho            datetime64[us]
VlEmpenhado                   float64
Liquidacao                        str
DataLiquidacao         datetime64[us]
VlLiquidado                   float64
OrdemPagamento                    str
DataPagamento          datetime64[us]
VlPago                        float64
NaturezaDespesa                   str
DsNaturezadaDespesa               str
CNPJ                              str
VlAnulado                     float64
Funcao                            str
SubFuncao                         str
FonteRecurso                      str
NmFavorecido                      str
Modalidade                        str
NumeroLicitacao                   str
arquivo_origem                    str
pago                             bool
dtype: object


In [13]:
print("Está bom quando: dtypes corretos e nenhum erro de conversao silencioso.\n")
print(base[COLUNAS_DATA + COLUNAS_VALOR].dtypes)
print()

erro_conversao = False
for c in COLUNAS_DATA:
    nat_depois = base[c].isna().sum()
    status = "OK" if nat_depois == nulos_antes[c] else "DIVERGENCIA"
    if nat_depois != nulos_antes[c]:
        erro_conversao = True
    print(f"{c}: nulos antes={nulos_antes[c]}, NaT depois={nat_depois} [{status}]")

for c in COLUNAS_VALOR:
    n_nan = base[c].isna().sum()
    status = "OK" if n_nan == 0 else "DIVERGENCIA"
    if n_nan != 0:
        erro_conversao = True
    print(f"{c}: NaN apos conversao para float={n_nan} [{status}]")

print(f"\nLinhas com VlPago = 0 (liquidado, ainda nao pago): {(~base['pago']).sum()} de {len(base)}")
assert not erro_conversao, "conversao introduziu nulos inesperados"
print("\nOK — todo NaT em data corresponde a um branco genuino na fonte; nenhum NaN inesperado em valor.")

Está bom quando: dtypes corretos e nenhum erro de conversao silencioso.

DataEmpenho       datetime64[us]
DataLiquidacao    datetime64[us]
DataPagamento     datetime64[us]
VlEmpenhado              float64
VlLiquidado              float64
VlPago                   float64
VlAnulado                float64
dtype: object

DataEmpenho: nulos antes=0, NaT depois=0 [OK]
DataLiquidacao: nulos antes=1769, NaT depois=1769 [OK]
DataPagamento: nulos antes=2491, NaT depois=2491 [OK]
VlEmpenhado: NaN apos conversao para float=0 [OK]
VlLiquidado: NaN apos conversao para float=0 [OK]


VlPago: NaN apos conversao para float=0 [OK]
VlAnulado: NaN apos conversao para float=0 [OK]

Linhas com VlPago = 0 (liquidado, ainda nao pago): 3532 de 12980

OK — todo NaT em data corresponde a um branco genuino na fonte; nenhum NaN inesperado em valor.


## Etapa 7 — Coluna derivada `grupo_gasto`

Regra (já validada no Sheets): se `NmFavorecido` for o banco operador da folha de pagamento,
`'Pessoal'`; senão, `'Custeio'`.

O favorecido exato foi confirmado cruzando com `DsNaturezadaDespesa`: as linhas de
`NmFavorecido == 'ITAU UNIBANCO S.A.'` concentram naturezas de despesa de folha
(`VENCIMENTOS E VANTAGENS FIXAS - PESSOAL CIVIL`, `TERCEIRIZACAO DE MAO DE OBRA`,
`OBRIGACOES PATRONAIS`, `AUXILIO - TRANSPORTE`, `AUXILIO ALIMENTACAO`, etc.). Outros bancos
aparecem no dataset (`BANCO DO BRASIL S/A`, `CAIXA ECONOMICA FEDERAL`) mas em naturezas de
despesa não relacionadas à folha (indenizações, obrigações patronais pontuais) — não entram na
regra, mantendo a leitura consistente com o número já validado no Sheets (~47%).

In [14]:
BANCO_FOLHA = "ITAU UNIBANCO S.A."

base["grupo_gasto"] = base["NmFavorecido"].apply(
    lambda x: "Pessoal" if x == BANCO_FOLHA else "Custeio"
)

print(base["grupo_gasto"].value_counts())

grupo_gasto
Custeio    11871
Pessoal     1109
Name: count, dtype: int64


In [15]:
pago_2025 = base[base["arquivo_origem"].str.contains("2025")]
total_pago_2025 = pago_2025["VlPago"].sum()
pessoal_2025 = pago_2025.loc[pago_2025["grupo_gasto"] == "Pessoal", "VlPago"].sum()
pct_pessoal = pessoal_2025 / total_pago_2025 * 100

print("Está bom quando: soma de VlPago para 'Pessoal' bate com ~R$ 1,07 bi / ~47% (2025).\n")
print(f"VlPago total pago 2025: R$ {total_pago_2025:,.2f}")
print(f"VlPago 'Pessoal' 2025: R$ {pessoal_2025:,.2f} ({pct_pessoal:.1f}% do total)")

assert 1.0e9 < pessoal_2025 < 1.15e9, "valor de Pessoal fora da faixa esperada (~R$ 1,07 bi)"
assert 44 < pct_pessoal < 50, "percentual de Pessoal fora da faixa esperada (~47%)"
print("\nOK — valor e percentual de 'Pessoal' batem com o achado ja documentado no README.")

Está bom quando: soma de VlPago para 'Pessoal' bate com ~R$ 1,07 bi / ~47% (2025).

VlPago total pago 2025: R$ 2,281,236,934.50
VlPago 'Pessoal' 2025: R$ 1,070,485,928.85 (46.9% do total)

OK — valor e percentual de 'Pessoal' batem com o achado ja documentado no README.


## Etapa 8 — Validação e export

Checagens finais antes de gravar a base tratada:

- total geral de `VlPago` em 2025 ≈ R$ 2,29 bi;
- nenhuma linha com `Empenho` nulo;
- contagem de linhas plausível (arquivos brutos menos rodapé e duplicatas).

Exporta para `data/clean/despesas_saude_2025_2026.csv`.

In [16]:
total_geral_2025 = base.loc[base["arquivo_origem"].str.contains("2025"), "VlPago"].sum()
empenhos_nulos = base["Empenho"].isna().sum()

print(f"Total VlPago 2025: R$ {total_geral_2025:,.2f} (esperado: ~R$ 2,29 bi)")
print(f"Linhas com Empenho nulo: {empenhos_nulos} (esperado: 0)")
print(f"Total de linhas na base final: {len(base)}")

assert 2.2e9 < total_geral_2025 < 2.4e9, "total pago 2025 fora da faixa esperada"
assert empenhos_nulos == 0, "ha linhas com Empenho nulo"

Total VlPago 2025: R$ 2,281,236,934.50 (esperado: ~R$ 2,29 bi)
Linhas com Empenho nulo: 0 (esperado: 0)
Total de linhas na base final: 12980


In [17]:
CAMINHO_SAIDA = CLEAN_DIR / "despesas_saude_2025_2026.csv"
base.to_csv(CAMINHO_SAIDA, index=False)
print(f"Base tratada exportada para: {CAMINHO_SAIDA.resolve()}")

Base tratada exportada para: C:\Users\nanda\analise-dados-saude\dados-saude\data\clean\despesas_saude_2025_2026.csv


In [18]:
conferencia = pd.read_csv(CAMINHO_SAIDA)

print("Está bom quando: o arquivo exportado abre sem erro e os numeros batem.\n")
print(f"Linhas no CSV exportado: {len(conferencia)} (esperado: {len(base)})")
print(f"Colunas no CSV exportado: {len(conferencia.columns)}")
print(f"Soma de VlPago no CSV re-lido: R$ {conferencia['VlPago'].sum():,.2f}")

assert len(conferencia) == len(base)
assert set(conferencia.columns) == set(base.columns)
print("\nOK — arquivo exportado reabre com a mesma contagem de linhas e colunas.")

Está bom quando: o arquivo exportado abre sem erro e os numeros batem.

Linhas no CSV exportado: 12980 (esperado: 12980)
Colunas no CSV exportado: 22
Soma de VlPago no CSV re-lido: R$ 3,479,027,860.51

OK — arquivo exportado reabre com a mesma contagem de linhas e colunas.
